# 14 · Right report: build, HTML and Excel from a real run

This notebook rebuilds the reconciliation report of `dtrack_right.report` from the check results of a real run: the run folder of a local run (e.g. Notebook 15's `_work/15/runs/<run_id>`) or a distributed run on S3. It walks the pieces (`collect`, the HTML sections, the Excel sheets, the CSVs), then the full `build_report` and the `publish_report` / `fetch_report` round trip on an S3 scratch prefix.

It mirrors `tests/unit/dtrack_right/report/test_build.py`, `tests/unit/dtrack_right/report/test_html.py` and `tests/unit/dtrack_right/report/test_excel.py`, with the in-memory store and literal statistics replaced by a real run folder.

## How to approach

- **Prerequisites**: kernel = the repo `.venv` (openpyxl and pandas are base dependencies). For an S3 run, AWS credentials that can read the run and write `DTRACK_PROD_S3` (`AWS_PROFILE` or the project `.env`). No Databricks or Left connection is needed.
- **Pick a run**: set `RUN_LOCATION` to a runs root or a single run folder, local path or `s3://...`. `resolve_run` picks the newest *unfinished* run under a root, so for a finished run also set `RUN_ID`. Run Notebook 15 first if you have no run yet.
- **Run order**: top to bottom. The run itself is only read; everything is written to `WORKDIR` and (section 8) to `S3_SCRATCH/nb14-<tag>/`. Section 7 modifies a *local copy* only.
- **What to look at**: the rendered HTML pieces in sections 4 and 6, the re-derived verdicts in section 3 (all should agree with the stored ones), and the byte-identical fetch in section 8.
- **Failure modes**: `resolve_run -> (None, None)` = every run under the root is complete, set `RUN_ID`; `no row_check documents found` = the folder is not a run (or Stage 1 never wrote verdicts); `NoSuchKey` on `units/...` = the run's results were produced elsewhere and the unit payloads are missing.

In [ ]:
import os
from pathlib import Path

RUN_LOCATION = os.environ.get("DTRACK_NB_RUN", str(Path.home() / ".local/plans/nbs/_work/15/runs"))  # runs root or run folder, local or s3://
RUN_ID = None                                                                       # None = newest unfinished run under RUN_LOCATION
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")    # only S3 prefix written to
WORKDIR = Path.home() / ".local/plans/nbs/_work/14"                                 # report output folder
TITLE = "Reconciliation Report"                                                     # report title
KEEP = False                                                                        # True = keep scratch objects and WORKDIR

In [ ]:
# Load the project .env (AWS defaults) and open the run read-only.
import json
import uuid
from datetime import UTC, datetime

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import HTML, display

from dtrack_right.backends.store import open_store, resolve_run, run_is_complete

load_dotenv(os.environ.get("DTRACK_ENV_FILE") or find_dotenv(usecwd=True), override=False)
TAG = uuid.uuid4().hex[:8]
SCRATCH_NAME = f"nb14-{TAG}"
WORKDIR.mkdir(parents=True, exist_ok=True)

# A runs root lists run folders; resolve_run picks RUN_ID or the newest unfinished one.
root = open_store(RUN_LOCATION)
names = root.list_prefixes()
print("runs under location:", {name: run_is_complete(root.child(name)) for name in names[-10:]})
store, run_id = resolve_run(RUN_LOCATION, RUN_ID)
if store is None:
    raise RuntimeError("every run under RUN_LOCATION is complete: set RUN_ID to one of the names above")
print("using run", run_id, "at", store.root_uri, "| complete:", run_is_complete(store))

## 1. What the report reads: collect()

`collect(store)` gathers the reconciled `row_check/<pair>.json` documents (skipping `.left.json` and `_CLOSED`), every `results/` and `dead/` entry, the manifests (skipping `.keys.json`) and `manifest/_CLOSED`.

In [ ]:
from dtrack_right.report.build import _summary, collect

state = collect(store)
print({k: len(v) if isinstance(v, (list, dict)) else v for k, v in state.items()})
display(pd.DataFrame([{"pair": d["pair"], **d["summary"], "gate": d["gate_decision"].get("passed"), "bounds": d.get("bounds")} for d in state["documents"]]))

# The run-level summary the report header uses: status, findings, units expected/completed.
summary_now = _summary(run_id, state, datetime.now(UTC))
print(json.dumps(summary_now, indent=2))
documents = state["documents"]
results_by_pair = {}
for row in state["results"]:
    results_by_pair.setdefault(row["pair"], []).append(row)

## 2. The CSVs: row check and column comparison

`row_check_csv_bytes` writes `pair,partition,left,right,delta,verdict`; `compare_col_csv_bytes` unions the fields of every result row and sorts by vintage then column.

In [ ]:
import io

from dtrack_right.checks.row_check import row_check_csv_bytes
from dtrack_right.report.build import compare_col_csv_bytes

for document in documents:
    pair = document["pair"]
    row_csv = row_check_csv_bytes(document)
    (WORKDIR / f"{pair}_compare_row.csv").write_bytes(row_csv)
    print(f"--- {pair}: row CSV, {len(document['partitions'])} partition(s)")
    display(pd.read_csv(io.BytesIO(row_csv)).query("verdict != 'match'").head(20))
    rows = results_by_pair.get(pair) or []
    if rows:
        col_csv = compare_col_csv_bytes(rows)
        (WORKDIR / f"{pair}_compare_col.csv").write_bytes(col_csv)
        frame = pd.read_csv(io.BytesIO(col_csv))
        print(f"col CSV: {len(frame)} row(s), {len(frame.columns)} field(s); diffs:")
        display(frame[frame["verdict"] == "diff"][["vintage", "column", "col_type", "n_total_diff", "n_unique_diff", "hash_sum_match", "mean_match"]].head(20))

## 3. Re-derive the verdicts with compare_statistics

Each result came from `compare_statistics(left=<unit payload>, right=<Right stats>)`. The Left payload is still in `units/<unit_id>.json` and the Right values are in the result's `*_right` fields, so the verdict can be recomputed and compared with the stored one.

In [ ]:
from dtrack_right.checks.col_check import compare_statistics


# Right-side statistics as compare_statistics expects them, read back from a stored result row.
def right_from_result(row):
    return {
        "col_type": row.get("col_type"),
        "n_total": row["n_total_right"],
        "n_missing": row["n_missing_right"],
        "n_unique": row["n_unique_right"],
        **{name: row.get(f"{name}_right") for name in ("mean", "std", "length_avg", "length_max", "hash_sum")},
        "min_val": row.get("min_right"),
        "max_val": row.get("max_right"),
        "top_10": row.get("top_10_right"),
    }


checked = []
for row in state["results"]:
    key = f"units/{row['unit_id']}.json"
    if not store.exists(key):
        continue
    again = compare_statistics(pair=row["pair"], window=row["vintage"], column=row["column"], left=store.get_json(key), right=right_from_result(row))
    checked.append({"pair": row["pair"], "vintage": row["vintage"], "column": row["column"], "stored": row["verdict"], "recomputed": again["verdict"]})
checked = pd.DataFrame(checked)
print(f"{len(checked)} result(s) re-derived; disagreements:", int((checked["stored"] != checked["recomputed"]).sum()) if len(checked) else 0)
display(checked.head(20))

## 4. HTML pieces

Scalar helpers (`format_count`, `_e`, `_seconds`, `_fmt`) on real numbers, then one `row_count_section` per pair (mismatched / one-side-only dates, at most ten listed each) and one `column_stats_section` per pair with results (diff columns per vintage, `left → right` names).

In [ ]:
from dtrack_right.report import html

# Real totals and timings through the formatting helpers.
for document in documents:
    total_left = sum(int(p["left"] or 0) for p in document["partitions"].values())
    timing = document.get("timing") or {}
    print(document["pair"], "| left rows:", html.format_count(total_left), "| left count:", html._seconds(timing.get("left_count_s")), "| right count:", html._seconds(timing.get("right_count_s")), "| escaped table:", html._e(document["left"].get("table")))
some = next(iter(state["results"]), None)
if some:
    print("_fmt examples on a real result:", {s: html._fmt(some.get(f"{s}_left"), s) for s in ("mean", "std", "hash_sum", "length_avg")})

row_sections, col_sections = [], []
for document in documents:
    row_sections.append(html.row_count_section(document))
    rows = results_by_pair.get(document["pair"]) or []
    if rows:
        labels = (str(document["left"].get("source") or "left"), str(document["right"].get("source") or "databricks"))
        col_sections.append(html.column_stats_section(document["pair"], rows, *labels, (document["left"].get("date_col"), document["right"].get("date_col"))))
print(f"{len(row_sections)} row section(s), {len(col_sections)} column section(s)")

In [ ]:
# The summary banner, the two tables and the page wrapper; written to WORKDIR and shown inline.
vintage = next(iter(state["manifests"].values()), {}).get("vintage", "month")
sections = [html.summary_section(summary_now), html.row_count_table(row_sections)]
if col_sections:
    sections.append(html.column_stats_table(col_sections, vintage=vintage))
page = html.wrap_document(TITLE, sections, subtitle=f"RUN {run_id} (pieces)")
(WORKDIR / "pieces.html").write_text(page, encoding="utf-8")
print("wrote", WORKDIR / "pieces.html", len(page), "chars")
display(HTML(page))

## 5. Excel pieces

The cell helpers strip control characters (`_clean`), keep hash sums as text (`_number`) and flag differing fields (`_differs`); `_relation` names a table or falls back to the path. `build_workbook` writes a `<pair>_row` sheet (findings first, red fill) and a `<pair>_col` sheet (PASS/FAIL per vintage) per pair.

In [ ]:
from openpyxl import load_workbook

from dtrack_right.report import excel

for document in documents:
    print(document["pair"], "| left:", excel._relation(document["left"]), "| right:", excel._relation(document["right"]))
if some:
    fields = ("n_total", "n_missing", "n_unique", "mean", "std", "hash_sum", "col_type")
    print("_differs on a real result:", {f: excel._differs(some, f) for f in fields})
    print("_number:", {f: excel._number(some.get(f"{f}_left"), f) for f in ("mean", "hash_sum")}, "| _clean:", excel._clean(str(some.get("min_left"))))

workbook = excel.build_workbook(documents, results_by_pair)
workbook.save(WORKDIR / "pieces.xlsx")
loaded = load_workbook(WORKDIR / "pieces.xlsx")
print("sheets:", loaded.sheetnames)
for sheet in loaded.worksheets:
    values = [[cell.value for cell in row] for row in sheet.iter_rows(max_row=16)]
    print(f"--- {sheet.title} ({sheet.max_row} rows)")
    display(pd.DataFrame(values).fillna(""))
    red = [cell.coordinate for row in sheet.iter_rows() for cell in row if cell.fill and str(cell.fill.start_color.rgb).endswith("FFC7CE")]
    print("red cells:", red[:20], "..." if len(red) > 20 else "")

## 6. The whole report: build_report

`build_report(store, out_dir)` writes `summary.json`, the per-pair CSVs, `index.html` and `report.xlsx` in one go. If the run already has its own `report/summary.json`, the findings are compared.

In [ ]:
from dtrack_right.report import build_report

files = build_report(store, WORKDIR / "report", title=TITLE)
print({name: path.stat().st_size for name, path in files.items()})
summary = json.loads(files["summary.json"].read_text())
print(json.dumps({k: summary[k] for k in ("status", "pairs", "findings", "units_expected", "units_completed", "errors")}, indent=2))
if store.exists("report/summary.json"):
    original = store.get_json("report/summary.json")
    print("run's own report:", original["status"], original["findings"], "| same findings:", original["findings"] == summary["findings"])
print("open:", files["index.html"].resolve().as_uri())

## 7. What-if variants on a local copy

The status rules: `PARTIAL` with dead units or without `manifest/_CLOSED`; without results the page has no Stage 2; an empty store is an error. The run's JSON is copied to `WORKDIR` and edited there (the injected dead unit is a labelled demo), the real run is never touched.

In [ ]:
from dtrack_right.backends.store import VolumeStore


# Copy the JSON the report reads into a fresh local VolumeStore.
def local_copy(name):
    copy_store = VolumeStore(WORKDIR / "whatif" / name)
    for key in store.list_keys(""):
        if key.split("/")[0] in {"row_check", "manifest", "results", "dead", "units"}:
            copy_store.put_bytes(key, store.get_bytes(key))
    return copy_store


def status_of(copy_store, name):
    built = build_report(copy_store, WORKDIR / "whatif" / f"{name}-report")
    built_summary = json.loads(built["summary.json"].read_text())
    has_stage2 = "Stage 2" in built["index.html"].read_text(encoding="utf-8")
    return built_summary["status"], built_summary["units_completed"], built_summary["errors"][:1], has_stage2


dead = local_copy("dead")
dead.put_dead("nb14-demo", {"unit_id": "nb14-demo", "error": "demo: injected by notebook 14"})
print("with a dead unit      :", status_of(dead, "dead"))

unclosed = local_copy("unclosed")
(unclosed.root / "manifest" / "_CLOSED").unlink(missing_ok=True)
print("without _CLOSED       :", status_of(unclosed, "unclosed"))

no_results = local_copy("no_results")
import shutil
shutil.rmtree(no_results.root / "results", ignore_errors=True)
print("without results       :", status_of(no_results, "no_results"))

try:
    build_report(VolumeStore(WORKDIR / "whatif" / "empty"), WORKDIR / "whatif" / "empty-report")
except ValueError as exc:
    print("empty store           :", exc)

## 8. Publish and fetch on the S3 scratch prefix

`publish_report` writes `history/<run_id>/<file>` (content type per extension) and then `latest/pointer.json` last; `fetch_report` follows the pointer and downloads the set. The fetched files should be byte-identical.

In [ ]:
from dtrack_left.backends.store import fetch_report
from dtrack_right.backends.store import S3Store
from dtrack_right.report import publish_report
from dtrack_right.report.build import CONTENT_TYPES

scratch = f"{S3_SCRATCH.rstrip('/')}/{SCRATCH_NAME}"
report_store = S3Store(f"{scratch}/report")
keys = publish_report(report_store, run_id, files, summary)
print("published:", keys)
print("pointer:", report_store.get_json("latest/pointer.json"))
for key in keys[:-1]:
    head = report_store.client.head_object(Bucket=report_store.bucket, Key=report_store._key(key))
    print(f"  {key:60} {head['ContentType']}  (expected {CONTENT_TYPES.get(Path(key).suffix, 'application/octet-stream')})")

fetched = fetch_report(report_store, WORKDIR / "fetched")
print("fetched run:", fetched["run_id"], sorted(fetched["files"]) == sorted(files))
print("byte-identical:", {name: fetched["files"][name].read_bytes() == path.read_bytes() for name, path in files.items()})

## Cleanup

Deletes `S3_SCRATCH/nb14-<tag>/` and `WORKDIR` unless `KEEP = True`. The run at `RUN_LOCATION` is never modified.

In [ ]:
# Delete everything this notebook wrote under the scratch prefix (and the local WORKDIR) unless KEEP.
import shutil

import boto3

scratch = f"{S3_SCRATCH.rstrip('/')}/{SCRATCH_NAME}"
if KEEP:
    print("KEEP=True: left", scratch, "and", WORKDIR)
else:
    bucket, _, prefix = scratch.removeprefix("s3://").partition("/")
    s3 = boto3.client("s3", endpoint_url=os.getenv("DTRACK_S3_ENDPOINT_URL") or None)
    deleted = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix + "/"):
        keys = [{"Key": row["Key"]} for row in page.get("Contents") or []]
        if keys:
            s3.delete_objects(Bucket=bucket, Delete={"Objects": keys})
            deleted += len(keys)
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print(f"deleted {deleted} object(s) under {scratch} and {WORKDIR}")

## What to check

- Section 1: `status` and `findings` look right for the run (DONE vs PARTIAL, counts of diffs).
- Section 3: zero disagreements between stored and recomputed verdicts.
- Sections 4-5: the HTML lists mismatched dates and diff columns as expected; Excel has `<pair>_row` / `<pair>_col` sheets, red cells on findings.
- Section 6: `build_report` findings equal the run's own report.
- Section 7: dead unit or missing `_CLOSED` gives PARTIAL; no results means no Stage 2; empty store raises.
- Section 8: content types match, pointer written last, fetched files byte-identical.